In [23]:
import numpy as np
import pandas as pd
import re

In [24]:
filename_template = R'template.csv'
filename_data = R'input.xlsx'

In [25]:
template = pd.read_csv(filename_template, on_bad_lines='skip')
data = pd.read_excel(filename_data, skiprows=1)

C:\Users\667016\AppData\Local\Temp\ipykernel_27140\150177265.py:1: DtypeWarning: Columns (0: Package Weight (g), 1: Serving Weight (g)) have mixed types. Specify dtype option on import or set low_memory=False.
  template = pd.read_csv(filename_template, on_bad_lines='skip')


In [26]:
def get_units(df):
    units = {}
    for column in df.columns:
        match = re.match(r'^(.*\S) ?\(([^()]+)\)$', column)
        if match:
            [col, unit] = match.groups()
            units[col] = unit
    return units

units = get_units(template)
units_origin = get_units(data)

In [27]:
unitors = { # value of key = 1 normalized unit (%/kcal/g)
    '%': 1,
    'kcal': 1,
    'g': 1, 
    'mg': 1_000,
    'ug': 1_000_000,
    'IU': 1 / 0.000_000_3 # approx.
}

In [28]:
def empties(df, cols):
    for col in cols:
        df[col] = ''
    return df


def matches(df, cols):
    for col in cols:
        origins = cols[col]
        unit = units[col]
        if not origins:
            df[f'{col} ({unit})'] = ''
            continue
        if not isinstance(origins, list):
            origins = [origins]
        sum = 0 # unit: g
        for origin in origins:
            unit_origin = units_origin[origin]
            normalized = df[f'{origin}({unit_origin})'] / unitors[unit_origin]
            sum += normalized
            # if unit == unit_origin:
            #     times = 1
            # elif unit == f"m{unit_origin}": # Original g transformed to mg
            #     times = 1000
            # elif f"m{unit}" == unit_origin: # Original mg transformed to g
            #     times = 0.001
            # elif unit == f"u{unit_origin}": # Original g transformed to ug
            #     times = 1000000
            # elif f"u{unit}" == unit_origin: # Original ug transformed to g
            #     times = 0.000001
            # else:
            #     raise TypeError()
            # sum += df[f'{origin}({unit_origin})'] * times
        df[f'{col} ({unit})'] = sum * unitors[unit]
    return df

def slashes(col):
    return col.str.replace(',', '/')

def replaces(col):
    return col.str.replace(',', '，').str.replace(':', '：').str.replace(';', '；')

In [29]:
units_origin

{'廢棄率': '%',
 '熱量': 'kcal',
 '修正熱量': 'kcal',
 '水分': 'g',
 '粗蛋白': 'g',
 '粗脂肪': 'g',
 '飽和脂肪': 'g',
 '灰分': 'g',
 '總碳水化合物': 'g',
 '膳食纖維': 'g',
 '糖質總量': 'g',
 '葡萄糖': 'g',
 '果糖': 'g',
 '半乳糖': 'g',
 '麥芽糖': 'g',
 '蔗糖': 'g',
 '乳糖': 'g',
 '鈉': 'mg',
 '鉀': 'mg',
 '鈣': 'mg',
 '鎂': 'mg',
 '鐵': 'mg',
 '鋅': 'mg',
 '磷': 'mg',
 '銅': 'mg',
 '錳': 'mg',
 '維生素A總量': 'IU',
 '視網醇當量(RE)': 'ug',
 '視網醇': 'ug',
 'α-胡蘿蔔素': 'ug',
 'β-胡蘿蔔素': 'ug',
 '維生素D總量': 'ug',
 '維生素D2': 'ug',
 '維生素D3': 'ug',
 '維生素E總量': 'mg',
 'α-維生素E當量(α-TE)': 'mg',
 'α-生育酚': 'mg',
 'β-生育酚': 'mg',
 'γ-生育酚': 'mg',
 'δ-生育酚': 'mg',
 '維生素K1': 'ug',
 '維生素K2 (MK-4)': 'ug',
 '維生素K2 (MK-7)': 'ug',
 '維生素B1': 'mg',
 '維生素B2': 'mg',
 '菸鹼素': 'mg',
 '維生素B6': 'mg',
 '維生素B12': 'ug',
 '葉酸': 'ug',
 '維生素C': 'mg',
 '脂肪酸S總量': 'mg',
 '酪酸(4:0)': 'mg',
 '己酸(6:0)': 'mg',
 '辛酸(8:0)': 'mg',
 '癸酸(10:0)': 'mg',
 '月桂酸(12:0)': 'mg',
 '十三酸(13:0)': 'mg',
 '肉豆蔻酸(14:0)': 'mg',
 '十五酸(15:0)': 'mg',
 '棕櫚酸(16:0)': 'mg',
 '十七酸(17:0)': 'mg',
 '硬脂酸(18:0)': 'mg',
 '十九酸(19:0)': 'mg',
 '花生

In [30]:
units

{'Package Weight': 'g',
 'Serving Weight': 'g',
 'Proteins': 'g',
 'Carbohydrates': 'g',
 'Energy': 'kcal',
 'Fats': 'g',
 'Saturated Fats': 'g',
 'Trans Fats': 'g',
 'Monounsaturated Fats': 'g',
 'Polyunsaturated Fats': 'g',
 'Omega-3': 'g',
 'Omega-6': 'g',
 'Sugars': 'g',
 'Added Sugars': 'g',
 'Dietary Fiber': 'g',
 'Soluble Fiber': 'g',
 'Insoluble Fiber': 'g',
 'Salt': 'g',
 'Cholesterol': 'g',
 'Caffeine': 'g',
 'Vitamin A': 'g',
 'Vitamin B1': 'g',
 'Vitamin B2': 'g',
 'Vitamin B3': 'g',
 'Vitamin B5': 'g',
 'Vitamin B6': 'g',
 'Vitamin B7': 'g',
 'Vitamin B9': 'g',
 'Vitamin B12': 'g',
 'Vitamin C': 'g',
 'Vitamin D': 'g',
 'Vitamin E': 'g',
 'Vitamin K': 'g',
 'Manganese': 'g',
 'Magnesium': 'g',
 'Potassium': 'g',
 'Calcium': 'g',
 'Copper': 'g',
 'Zinc': 'g',
 'Sodium': 'g',
 'Iron': 'g',
 'Phosphorus': 'g',
 'Selenium': 'g',
 'Iodine': 'g',
 'Chromium': 'g'}

In [31]:
data['俗名'] = slashes(data['俗名'].fillna(''))
data['Name'] = data.apply(lambda row: row['樣品名稱'] if row['俗名'] == '' else f"{row['樣品名稱']}({row['俗名']})", axis=1)
data['Note'] = data['食品分類'] + "；" + replaces(data['內容物描述'])
data['Is Liquid'] = 0
data['Source URL'] = 'https://consumer.fda.gov.tw/Food/TFND.aspx?nodeID=178 @2026.04.30'
data = empties(data, [
    'Brand', 
    'Barcode'
])
data = matches(data, {
    'Proteins': '粗蛋白',
    'Carbohydrates': '總碳水化合物',
    'Package Weight': None, 
    'Serving Weight': None, 
    'Energy': '熱量',
    'Fats': '粗脂肪',
    'Saturated Fats': '飽和脂肪',
    'Trans Fats': '反式脂肪',
    'Monounsaturated Fats': '脂肪酸M總量',
    'Polyunsaturated Fats': '脂肪酸P總量',
    'Omega-3': ['次亞麻油酸(18:3)', '廿碳五烯酸(20:5)', '廿二碳五烯酸(22:5)', '廿二碳六烯酸(22:6)'],
    'Omega-6': ['亞麻油酸(18:2)', '花生油酸(20:4)'],
    'Sugars': '糖質總量',
    'Added Sugars': None,
    'Dietary Fiber': '膳食纖維',
    'Soluble Fiber': None,
    'Insoluble Fiber': None,
    'Salt': None,
    'Cholesterol': '膽固醇',
    'Caffeine': None,
    'Vitamin A': '維生素A總量',
    'Vitamin B1': '維生素B1',
    'Vitamin B2': '維生素B2',
    'Vitamin B3': '菸鹼素',
    'Vitamin B5': None,
    'Vitamin B6': '維生素B6',
    'Vitamin B7': None,
    'Vitamin B9': '葉酸',
    'Vitamin B12': '維生素B12',
    'Vitamin C': '維生素C',
    'Vitamin D': '維生素D總量',
    'Vitamin E': '維生素E總量',
    'Vitamin K': '維生素K1',
    'Manganese': '錳',
    'Magnesium': '鎂',
    'Potassium': '鉀',
    'Calcium': '鈣',
    'Copper': '銅',
    'Zinc': '鋅',
    'Sodium': '鈉',
    'Iron': '鐵',
    'Phosphorus': '磷',
    'Selenium': None,
    'Iodine': None,
    'Chromium': None
})

C:\Users\667016\AppData\Local\Temp\ipykernel_27140\2652783819.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Name'] = data.apply(lambda row: row['樣品名稱'] if row['俗名'] == '' else f"{row['樣品名稱']}({row['俗名']})", axis=1)
C:\Users\667016\AppData\Local\Temp\ipykernel_27140\2652783819.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Note'] = data['食品分類'] + "；" + replaces(data['內容物描述'])
C:\Users\667016\AppData\Local\Temp\ipykernel_27140\2652783819.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usua

In [32]:
data[['Name', 'Omega-3 (g)', 'Vitamin A (g)']]

,Name,Omega-3 (g),Vitamin A (g)
0,大麥仁(小薏仁/洋薏仁/珍珠薏仁),0.054941,0.000000
1,大麥片,0.063262,0.000000
2,大麥仁粉(小薏仁粉/洋薏仁粉/珍珠薏仁粉),0.041949,0.000000
3,小米(狐尾粟/谷子/粟/粱/稷),0.034891,0.000000
4,小米(2025年取樣)(狐尾粟/谷子/粟/粱/稷),0.045673,0.000007
...,...,...,...
2208,啤酒,NaN,0.000000
2209,陳年紹興酒,NaN,0.000000
2210,白葡萄酒,NaN,0.000000
2211,紅葡萄酒,NaN,0.000000


In [33]:
data[template.columns].to_csv(R'output.foodyou.csv', encoding='utf-8', index=False)